# 03 · 벡터DB 벤치마크 (Phase 5)

pgvector / Qdrant / Neo4j 를 **동일 문서·동일 질의**로 비교한다.

- **Round 1** — 임베딩 고정(`dragonkue/BGE-m3-ko`), DB 3종 비교
- **Round 2** — DB 고정(Round 1 우승), 임베딩 3종 비교

측정: 인덱싱 시간 · 검색 latency(p50/p95) · 검색 품질(hit_rate@k, mrr@k) · (선택) RAGAS context_precision/recall.

> 설계: **임베딩과 DB를 분리**한다. 코퍼스·질의 임베딩을 한 번 계산해 재사용하고, 백엔드에는 raw 벡터만 넣는다 → 검색 latency에 임베딩 비용이 섞이지 않는다.

> 사전조건: `docker-compose up -d` 로 pgvector·qdrant·neo4j 컨테이너가 떠 있어야 한다. 생성 지표(faithfulness/answer_relevancy)는 Phase 5에서 제외한다 — DB·임베딩만 바뀌고 생성 LLM은 고정이라 변별력이 없다.

## 1. 설정 & 임포트

In [6]:
import os
import sys

# 프로젝트 루트를 path에 추가 (노트북 cwd 무관)
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from src import bench, chunker, config, dataset, embedder, evaluator, retrieval_metrics
from src.backends import get_backend

K = config.TOP_K
golden = dataset.load_golden_set()
questions = [g['question'] for g in golden]
print(f'골든셋 {len(golden)}개 · top_k={K}')

골든셋 50개 · top_k=4


## 2. 코퍼스 청킹 (DB·임베딩 공통 입력)

빠른 확인은 `MAX_DOCS`를 작게 잡는다. 진짜 벤치마크는 `None`(전체)로 둔다.

In [7]:
MAX_DOCS = None  # 전체로 측정하려면 None

documents = chunker.load_documents(config.DOCS_DIR, max_docs=MAX_DOCS)
chunks = chunker.chunk_documents(documents)
texts = [c.page_content for c in chunks]
titles = [c.metadata.get('title') for c in chunks]
print(f'문서 {len(documents)}개 → 청크 {len(chunks)}개')

문서 1417개 → 청크 13808개


### 한 조합 측정 헬퍼

`(백엔드, 임베딩 벡터)` 하나를 받아 인덱싱·검색·지표를 한 행(dict)으로 돌려준다.
Round 1은 임베딩을 1회만 계산해 3개 DB에 재사용하고, Round 2는 임베딩마다 새로 계산한다.

In [8]:
def bench_one(backend_name, vectors, qvectors, *, embed_corpus_s=None,
              embed_model=None, collection=config.BENCH_COLLECTION):
    """(백엔드 1개 + 임베딩 벡터)를 받아 한 조합을 측정 → (결과표 한 행, 검색결과)를 반환.

    Round 1은 DB만 바꿔 3번, Round 2는 임베딩만 바꿔 3번 호출해 표를 채운다.
    vectors=청크 임베딩, qvectors=골든 질문 임베딩. `*` 뒤는 키워드 전용 인자.
    """
    dim = bench.embedding_dim(vectors)              # 임베딩 차원(예: BGE-m3-ko=1024) — 테이블/컬렉션 생성에 필요
    be = get_backend(backend_name, collection=collection)  # 팩토리로 백엔드 생성 → DB 커넥션 오픈
    try:
        # DB 비우기→적재→인덱스 구축까지의 소요 초(=인덱싱 시간). 임베딩은 제외(이미 계산됨)
        t_index = bench.index_corpus(be, dim, vectors, texts, titles)
        # 골든 질의를 top-K 검색. results=질의별 Hit 리스트, latencies=질의별 소요(ms)
        results, latencies = bench.run_queries(be, qvectors, K)
        n = be.count()                              # 실제 적재된 벡터 개수(적재 검증용)
    finally:
        be.close()                                  # 성공/실패 무관하게 커넥션 정리(누수 방지)

    # 결과표의 한 행을 dict로 구성: 백엔드·임베딩명·청크수·차원·인덱싱 시간
    row = {'backend': backend_name, 'embed_model': embed_model,
           'n_chunks': n, 'dim': dim, 'index_s': round(t_index, 2)}
    if embed_corpus_s is not None:                  # 임베딩 시간이 넘어왔으면 기록(Round 2의 핵심 지표)
        row['embed_corpus_s'] = round(embed_corpus_s, 2)
    # latency 분포 요약(p50/p95/mean ms)을 행에 합침
    row.update({k: round(v, 2) for k, v in bench.latency_stats(latencies).items()})
    # 검색 품질(hit_rate@K, mrr@K)을 행에 합침 — LLM 없이 정답 스팬 포함 여부로 판정
    row.update({k: round(v, 4) for k, v in
                retrieval_metrics.evaluate_retrieval(results, golden, K).items()})
    return row, results                             # 행은 표로 쌓이고, results는 RAGAS(셀 4·5)에서 재사용


## 3. Round 1 — 임베딩 고정, DB 3종 비교

동일 벡터를 pgvector / Qdrant / Neo4j 에 각각 적재·검색한다. 임베딩이 같으니 검색 품질(hit_rate/mrr)은 비슷해야 정상이고, **차이는 인덱싱 시간·latency에서 난다.**

In [9]:
embeddings = embedder.build_embeddings(config.EMBED_MODEL)
vectors, t_embed = bench.embed_corpus(embeddings, texts)
qvectors, _ = bench.embed_queries(embeddings, questions)
print(f'임베딩 {config.EMBED_MODEL}: dim={bench.embedding_dim(vectors)}, 코퍼스 {t_embed:.1f}s')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8730.23it/s]


임베딩 dragonkue/BGE-m3-ko: dim=1024, 코퍼스 1067.4s


In [10]:
rows_r1, results_r1 = [], {}
for name in config.BENCH_BACKENDS:
    print(f'▶ {name} 측정 중...')
    try:
        row, res = bench_one(name, vectors, qvectors,
                             embed_corpus_s=t_embed, embed_model=config.EMBED_MODEL)
        rows_r1.append(row); results_r1[name] = res
    except Exception as e:
        print(f'  ⚠️ {name} 실패 — 컨테이너 확인 필요: {e!r}')

df_r1 = pd.DataFrame(rows_r1)
df_r1

▶ pgvector 측정 중...
▶ qdrant 측정 중...
▶ neo4j 측정 중...


,backend,embed_model,n_chunks,dim,index_s,embed_corpus_s,latency_p50_ms,latency_p95_ms,latency_mean_ms,hit_rate@4,mrr@4
0,pgvector,dragonkue/BGE-m3-ko,13808,1024,20.24,1067.37,30.40,34.25,31.33,0.98,0.9367
1,qdrant,dragonkue/BGE-m3-ko,13808,1024,6.88,1067.37,2.52,9.47,4.36,0.96,0.9167
2,neo4j,dragonkue/BGE-m3-ko,13808,1024,14.92,1067.37,5.10,8.78,7.58,0.88,0.8233


In [12]:
df_r1.to_csv(config.BENCH_DB_PATH, index=False, encoding='utf-8-sig')

## 4. (선택) Round 1 RAGAS 검색 지표

judge LLM(Claude) 비용이 든다. 임베딩이 같으면 DB별 검색 결과가 거의 동일해 `context_precision/recall`도 비슷하므로, **대표로 1개 DB만** 평가해도 충분하다. Round 2(임베딩 비교)에서 본격적으로 쓴다.

In [ ]:
RUN_RAGAS_R1 = False  # True로 켜면 judge 호출
if RUN_RAGAS_R1 and results_r1:
    judge = evaluator.build_judge()
    ev_llm, ev_emb = evaluator.build_evaluators(judge, embeddings)
    name = next(iter(results_r1))
    recs = evaluator.build_retrieval_records(golden, results_r1[name])
    r = evaluator.run_ragas_retrieval(recs, ev_llm, ev_emb)
    print(name, r)

## 5. Round 2 — DB 고정, 임베딩 3종 비교

Round 1 우승 DB를 고정하고 임베딩만 바꾼다. 여기서는 검색 결과가 임베딩마다 **실제로 달라지므로** hit_rate/mrr과 RAGAS 지표가 의미를 갖는다. 임베딩 생성 속도도 함께 잰다.

> ⚠️ 임베딩 모델 3종을 처음 받을 때 다운로드가 오래 걸릴 수 있다.

In [16]:
import gc, torch
for n in ('embeddings','emb','vecs','qvecs','vectors'):
    globals().pop(n, None)
gc.collect(); torch.mps.empty_cache()


In [18]:
import importlib
from src import embedder
importlib.reload(embedder)


<module 'src.embedder' from '/Users/doyoung/playground/rag-eval-kit/src/embedder.py'>

In [20]:
import gc

WINNER_DB = 'qdrant'   # Round 1 결과를 보고 지정 (속도·품질 균형 우승)
RUN_RAGAS_R2 = False   # 검색 품질 비교가 핵심이면 RAGAS는 선택
RESUME = True          # True: 이미 끝난 모델 건너뛰고 이어하기 / False: 처음부터 다시
CKPT = config.BENCH_EMBED_PATH  # 모델 1개 끝날 때마다 여기에 append → 중간에 끊겨도 보존


def free_memory(*names):
    """전역에서 이름들의 참조를 끊고 GC + 디바이스 캐시를 비운다 (RAM/MPS 누적 방지).

    임베딩 모델은 수 GB라, 다음 모델을 올리기 전에 명시적으로 풀지 않으면 RAM이 누적된다.
    GC만으로는 MPS/CUDA 캐시가 반납되지 않으므로 empty_cache까지 호출한다.
    """
    g = globals()
    for n in names:
        g.pop(n, None)
    gc.collect()
    try:
        import torch
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


# 체크포인트 준비 — RESUME=False면 기존 결과를 지우고 처음부터
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not RESUME and CKPT.exists():
    CKPT.unlink()
done = set(pd.read_csv(CKPT)['embed_model']) if CKPT.exists() else set()
if done:
    print(f'⏭️  이미 완료(건너뜀): {sorted(done)}')

free_memory('embeddings')  # Round 1 BGE 모델이 남아 있으면 해제 (모델 1개분 RAM 확보)

judge = ev_llm = ev_emb = None
for model in config.BENCH_EMBED_MODELS:
    if model in done:
        continue
    print(f'▶ 임베딩 {model} ...')
    try:
        emb = embedder.build_embeddings(model, show_progress=True)  # 코퍼스 진행바 ON
        vecs, te = bench.embed_corpus(emb, texts)
        emb.show_progress = False                                   # 질의 50개는 진행바 OFF(노이즈)
        qvecs, _ = bench.embed_queries(emb, questions)
        # 인덱싱·검색은 벡터만 쓰므로, RAGAS를 안 쓰면 모델(GB)을 여기서 즉시 해제
        if not RUN_RAGAS_R2:
            free_memory('emb')
        row, res = bench_one(WINNER_DB, vecs, qvecs, embed_corpus_s=te,
                             embed_model=model, collection=config.BENCH_COLLECTION)
        if RUN_RAGAS_R2:
            if ev_llm is None:
                judge = evaluator.build_judge()
                ev_llm, ev_emb = evaluator.build_evaluators(judge, emb)
            recs = evaluator.build_retrieval_records(golden, res)
            rg = evaluator.run_ragas_retrieval(recs, ev_llm, ev_emb)
            row['context_precision'] = round(float(rg['context_precision']), 4)
            row['context_recall'] = round(float(rg['context_recall']), 4)
        # ✅ 모델 1개 끝나면 즉시 CSV에 append → 다음 모델에서 끊겨도 이 결과는 보존(이어하기)
        pd.DataFrame([row]).to_csv(CKPT, mode='a', header=not CKPT.exists(),
                                   index=False, encoding='utf-8-sig')
        print(f'  ✅ 저장: {model}  (index_s={row["index_s"]}, hit_rate@{K}={row[f"hit_rate@{K}"]})')
    except Exception as e:
        print(f'  ⚠️ {model} 실패: {e!r}')
    finally:
        # 다음 모델을 로드하기 전에 현재 모델·벡터를 모두 정리 → 피크 상주 = 모델 1개
        free_memory('emb', 'vecs', 'qvecs', 'res')

df_r2 = pd.read_csv(CKPT) if CKPT.exists() else pd.DataFrame()
df_r2


⏭️  이미 완료(건너뜀): ['Qwen/Qwen3-Embedding-0.6B', 'dragonkue/BGE-m3-ko']
▶ 임베딩 google/embeddinggemma-300m ...


Batches: 100%|██████████| 432/432 [11:38<00:00,  1.62s/it]


  ✅ 저장: google/embeddinggemma-300m  (index_s=5.72, hit_rate@4=0.82)


,backend,embed_model,n_chunks,dim,index_s,embed_corpus_s,latency_p50_ms,latency_p95_ms,latency_mean_ms,hit_rate@4,mrr@4
0,qdrant,dragonkue/BGE-m3-ko,13808,1024,7.71,1132.66,2.71,3.74,3.02,0.98,0.9367
1,qdrant,Qwen/Qwen3-Embedding-0.6B,13808,1024,6.54,1939.86,2.27,3.41,2.52,0.76,0.6783
2,qdrant,google/embeddinggemma-300m,13808,768,5.72,698.93,2.09,2.42,2.16,0.82,0.7433


## 6. 결과 저장 & 해석

In [21]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not df_r1.empty:
    df_r1.to_csv(config.BENCH_DB_PATH, index=False, encoding='utf-8-sig')
    print('저장:', config.BENCH_DB_PATH)
if not df_r2.empty:
    df_r2.to_csv(config.BENCH_EMBED_PATH, index=False, encoding='utf-8-sig')
    print('저장:', config.BENCH_EMBED_PATH)

저장: /Users/doyoung/playground/rag-eval-kit/results/bench_db.csv
저장: /Users/doyoung/playground/rag-eval-kit/results/bench_embedding.csv


### 해석 가이드

- **Round 1(DB 비교)**: 검색 품질(hit_rate/mrr)은 임베딩이 같아 비슷해야 한다. 차이가 크다면 거리 메트릭/인덱스 설정 불일치를 의심한다. 실질 비교축은 **index_s(인덱싱 시간)** 과 **latency_p95_ms**.
- **Round 2(임베딩 비교)**: hit_rate/mrr·RAGAS가 실제로 갈린다. `embed_corpus_s`(임베딩 속도)와 검색 품질의 트레이드오프를 본다.
- 관련성 판정은 *정답 스팬이 청크에 포함되는가*(결정론적)다. judge LLM 없이도 DB·임베딩 비교에 충분한 신호를 준다.

결과 표는 README 비교표(Phase 6)로 옮긴다.